# Originality Scoring

利用OpenScoring的接口，对AUT用途回答进行LLM评分。

In [ ]:
import pandas as pd
import requests
import time
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
# 输入文件路径
data_dir = Path('../../data/preprocessed')
responses_file = data_dir / 'responses.csv'

# 输出文件路径
output_dir = Path('../../data/analysis/scoring')
originality_file = output_dir / 'originality.csv'
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 加载数据
responses = pd.read_csv(responses_file, encoding='utf-8-sig')
# 筛选用户回答
user_msgs = responses[responses['role'] == 'user'].copy().reset_index(drop=True)
print(f"共 {len(user_msgs)} 条用户回答需要评分")

In [ ]:
# API 配置
API_URL = "https://openscoring.du.edu/llm"
HEADERS = {
    "accept": "application/json",
    "content-type": "application/x-www-form-urlencoded",
}
# 固定参数
MODEL = "ocsai2"    # 选取的评分LLM模型
LANGUAGE = "chi"    # 语言设置为中文
TASK = "uses"       # 评分任务类型，"uses"表示物品用途评分
ELAB_METHOD = "none"
LOGPROB_SCORING = "true"

# 分批大小（避免请求过大）
BATCH_SIZE = 500
# 单次请求超时时间（秒）
REQUEST_TIMEOUT = 120

In [ ]:
def build_input_batch(rows):
    """从DataFrame行构建input字符串，每行格式：prompt,response"""
    lines = []
    for _, row in rows.iterrows():
        response = clean_response(row['content'])
        lines.append(f"{row['item_name']},{response}")
    return "\n".join(lines)

def call_api(input_text):
    """发送POST请求，返回JSON响应"""
    # 构建请求体
    data = {
        "input": input_text,
        "model": MODEL,
        "language": LANGUAGE,
        "task": TASK,
        "elab_method": ELAB_METHOD,
        "logprob_scoring": LOGPROB_SCORING
    }
    # 使用requests库，自动编码为x-www-form-urlencoded
    response = requests.post(API_URL, headers=HEADERS, data=data, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()
    
def clean_response(text):
    """清洗回答内容，避免干扰分隔符"""
    if pd.isna(text):
        return ''
    # 替换英文逗号为中文逗号（防止被误当作分隔符）
    text = text.replace(',', '，')
    # 替换换行符为空格
    text = text.replace('\n', ' ').replace('\r', ' ')
    # 替换制表符为空格
    text = text.replace('\t', ' ')
    return text

In [ ]:
# 预先创建结果列，确保中途失败时也能保留已完成批次
user_msgs['originality'] = np.nan

# 如果已有历史结果，先回填，支持按批次续跑
if originality_file.exists():
    previous = pd.read_csv(originality_file, encoding='utf-8-sig')
    if len(previous) == len(user_msgs) and 'originality' in previous.columns:
        user_msgs['originality'] = previous['originality']
        print(f"检测到已有结果文件，已回填 {user_msgs['originality'].notna().sum()} 条分数")
    else:
        print("检测到已有结果文件，但长度或字段不匹配，将重新从当前数据开始续跑")

# 分批处理
total_batches = (len(user_msgs) + BATCH_SIZE - 1) // BATCH_SIZE
for i in tqdm(range(0, len(user_msgs), BATCH_SIZE), total=total_batches, desc="批处理进度"):
    batch = user_msgs.iloc[i:i+BATCH_SIZE]
    if batch['originality'].notna().all():
        print(f"批次 {i // BATCH_SIZE + 1} 已完成，跳过")
        continue
    input_text = build_input_batch(batch)
    
    # 重试机制
    max_retries = 5
    for attempt in range(max_retries):
        try:
            result = call_api(input_text)
            if result and 'scores' in result:
                # 提取分数，顺序应与batch一致
                scores = result['scores']
                if len(scores) != len(batch):
                    print(f"警告：返回的分数数量({len(scores)})与提交数量({len(batch)})不一致")
                # 将分数合并到batch的索引中
                for j, score_item in enumerate(scores):
                    if j >= len(batch):
                        break
                    # 检查prompt和response是否匹配（可选）
                    # 如果API顺序错乱，可以根据内容再次匹配，但一般顺序一致
                    # 这里直接按顺序添加
                    user_msgs.loc[i + j, 'originality'] = score_item.get('originality', np.nan)
                # 每个批次成功后立即落盘，降低中断重跑成本
                user_msgs.to_csv(originality_file, index=False, encoding='utf-8-sig')
                break  # 成功则跳出重试循环
            else:
                print(f"API返回异常，尝试重试 {attempt+1}/{max_retries}")
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"请求异常：{e}，尝试重试 {attempt+1}/{max_retries}")
            time.sleep(2 ** attempt)
    else:
        # 所有重试均失败，标记为NaN
        print(f"批次 {i//BATCH_SIZE + 1} 最终失败，标记为缺失")
        print(f"批次输入：\n{input_text}\n")
        user_msgs.loc[i:i+len(batch)-1, 'originality'] = np.nan
        user_msgs.to_csv(originality_file, index=False, encoding='utf-8-sig')
    
    # 适当延时，避免请求过快
    time.sleep(2)

# 确保最终结果已保存
user_msgs.to_csv(originality_file, index=False, encoding='utf-8-sig')

# 检查缺失情况
missing = user_msgs['originality'].isna().sum()
if missing > 0:
    print(f"警告：有 {missing} 条回答未成功获取分数")
else:
    print("所有回答评分完成！")

# 保存结果
user_msgs.to_csv(originality_file, index=False, encoding='utf-8-sig')